# Notebook 06 — SMAP MAML-AE (Corrected Core, timeout-proof)

Corrected FOMAML (gradients from the adapted learner are transferred onto the meta-model),
compared against a properly-trained static AE under an identical few-shot protocol.

### ▶ How to run so it survives Kaggle timeouts
1. Attach your `phd-maml-ae-data` dataset as input. Turn the **accelerator to GPU**.
2. Use **Save Version → "Save & Run All (Commit)"** (NOT interactive run). A commit runs
   headless for up to ~9–12 h and **persists `/kaggle/working` as output** automatically.
   The full 30 000-step run is ~1–2 h on GPU, so it finishes in one commit — no resume needed.
3. **Only if a run is ever interrupted:** open the next version, also attach *this notebook's
   previous output* as an input, and Commit again. Startup auto-detects the newest checkpoint
   in any attached input and resumes from it. Nothing restarts from zero.


## 1 — Imports, paths, data, checkpoint discovery

In [1]:
import os, json, pickle, copy, time, warnings
import numpy as np
import torch, torch.nn as nn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
warnings.filterwarnings('ignore')
torch.manual_seed(42); np.random.seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

INPUT_PATH  = "/kaggle/input/datasets/abhijeetckarve/phd-maml-ae-data"  # <-- fix if os.walk shows a different path
OUTPUT_PATH = "/kaggle/working"
WORK_MODELS = f"{OUTPUT_PATH}/models"
os.makedirs(WORK_MODELS, exist_ok=True)
os.makedirs(f"{OUTPUT_PATH}/results", exist_ok=True)

# ---- data ----
with open(f"{INPUT_PATH}/channel_data_normalized.pkl","rb") as f:
    channel_data = pickle.load(f)
with open(f"{INPUT_PATH}/task_splits_25feat.json","r") as f:
    task_splits_25 = json.load(f)
meta_train_tasks = task_splits_25['meta_train']
meta_val_tasks   = task_splits_25['meta_val']
EVAL_CHANNELS    = [ch for ch in task_splits_25['meta_test'] if ch not in ['P-4','D-12']]
print(f"meta-train {len(meta_train_tasks)} | meta-val {len(meta_val_tasks)} | eval {EVAL_CHANNELS}")

# ---- checkpoint discovery: search all attached inputs + working, keep newest ----
def _all_search_dirs():
    dirs=[WORK_MODELS]
    for root,_,_ in os.walk("/kaggle/input"): dirs.append(root)
    return dirs

def resolve_ckpt(name, step_key="step"):
    '''Return (path, step) of the highest-step file called `name` across input+working, else (None,-1).'''
    best_path, best_step = None, -1
    for d in _all_search_dirs():
        p=os.path.join(d,name)
        if os.path.exists(p):
            try:
                step=torch.load(p, map_location="cpu").get(step_key, 0)
            except Exception:
                step=0
            if step>=best_step: best_step, best_path = step, p
    return best_path, best_step

CKPT_NAME="smap_maml_ckpt.pt"; BEST_NAME="smap_maml_best.pt"; STATIC_NAME="smap_static_ae.pt"
print("startup checkpoint scan:",
      "resume=", resolve_ckpt(CKPT_NAME)[1],
      "| best=", resolve_ckpt(BEST_NAME, 'step')[0] is not None,
      "| static=", resolve_ckpt(STATIC_NAME,'val_dummy')[0] is not None)


Device: cuda
meta-train 39 | meta-val 6 | eval ['E-3', 'D-7', 'E-6', 'D-6', 'T-2', 'A-6', 'D-3']
startup checkpoint scan: resume= -1 | best= False | static= False


## 2 — Model classes (verbatim from Notebook 4/5)

In [2]:
class LSTMEncoder(nn.Module):
    def __init__(self, input_size=25, hidden1=64, hidden2=32, latent=16):
        super().__init__()
        self.lstm1=nn.LSTM(input_size,hidden1,batch_first=True)
        self.lstm2=nn.LSTM(hidden1,hidden2,batch_first=True)
        self.fc=nn.Linear(hidden2,latent)
    def forward(self,x):
        out,_=self.lstm1(x); _,(h_n,_)=self.lstm2(out); return self.fc(h_n.squeeze(0))
class LSTMDecoder(nn.Module):
    def __init__(self, latent=16, hidden1=32, hidden2=64, output_size=25, seq_len=30):
        super().__init__()
        self.seq_len=seq_len
        self.lstm1=nn.LSTM(latent,hidden1,batch_first=True)
        self.lstm2=nn.LSTM(hidden1,hidden2,batch_first=True)
        self.fc=nn.Linear(hidden2,output_size)
    def forward(self,z):
        z_rep=z.unsqueeze(1).repeat(1,self.seq_len,1)
        out,_=self.lstm1(z_rep); out,_=self.lstm2(out); return self.fc(out)
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size=25, seq_len=30, hidden1=64, hidden2=32, latent=16):
        super().__init__()
        self.encoder=LSTMEncoder(input_size,hidden1,hidden2,latent)
        self.decoder=LSTMDecoder(latent,hidden2,hidden1,input_size,seq_len)
    def forward(self,x): return self.decoder(self.encoder(x))
    def reconstruction_error(self,x):
        x_hat=self.forward(x); return torch.mean((x-x_hat)**2,dim=(1,2))
class MLPAutoencoder(nn.Module):
    def __init__(self, input_size=25, seq_len=30, latent=16):
        super().__init__()
        self.seq_len,self.input_size=seq_len,input_size; flat=seq_len*input_size
        self.encoder=nn.Sequential(nn.Linear(flat,256),nn.ReLU(),nn.Linear(256,64),nn.ReLU(),nn.Linear(64,latent))
        self.decoder=nn.Sequential(nn.Linear(latent,64),nn.ReLU(),nn.Linear(64,256),nn.ReLU(),nn.Linear(256,flat))
    def forward(self,x):
        b=x.shape[0]; z=self.encoder(x.reshape(b,-1))
        return self.decoder(z).reshape(b,self.seq_len,self.input_size)
    def reconstruction_error(self,x):
        x_hat=self.forward(x); return torch.mean((x-x_hat)**2,dim=(1,2))
print("model classes ready")


model classes ready


## 3 — Corrected FOMAML core
**THE FIX** is in `outer_step`: `torch.autograd.grad` on the adapted learner, then those
grads are written onto the meta-model's `.grad` before stepping.

In [3]:
criterion = nn.MSELoss()
def sample_episode(channel_id, support_size=20, query_size=20, rng=np.random):
    w=channel_data[channel_id]['normal_windows']; idx=rng.permutation(len(w)); need=support_size+query_size
    if len(w)<need:
        s=w[rng.choice(len(w),support_size,replace=True)]; q=w[rng.choice(len(w),query_size,replace=True)]
    else:
        s=w[idx[:support_size]]; q=w[idx[support_size:need]]
    return (torch.tensor(s,dtype=torch.float32).to(DEVICE), torch.tensor(q,dtype=torch.float32).to(DEVICE))
def inner_adapt(model, support, inner_lr=0.01, inner_steps=10):
    learner=copy.deepcopy(model); learner.train()
    opt=torch.optim.SGD(learner.parameters(),lr=inner_lr)
    for _ in range(inner_steps):
        opt.zero_grad(); loss=criterion(learner(support),support); loss.backward(); opt.step()
    return learner
def outer_step(model, outer_opt, task_batch, inner_lr=0.01, inner_steps=10,
               support_size=20, query_size=20, train=True, rng=np.random):
    accum=[None]*len(list(model.parameters())); meta_loss=0.0
    for ch in task_batch:
        support,query=sample_episode(ch,support_size,query_size,rng)
        learner=inner_adapt(model,support,inner_lr,inner_steps)
        qloss=criterion(learner(query),query)
        if train:
            g=torch.autograd.grad(qloss,learner.parameters())
            accum=[gi.detach() if a is None else a+gi.detach() for a,gi in zip(accum,g)]
        meta_loss+=qloss.item()
    meta_loss/=len(task_batch)
    if train:
        outer_opt.zero_grad()
        for p,a in zip(model.parameters(),accum): p.grad=a/len(task_batch)
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); outer_opt.step()
    return meta_loss
print("corrected FOMAML core ready")


corrected FOMAML core ready


## 4 — Meta-train MAML (auto-resume from newest checkpoint anywhere)

In [4]:
INNER_LR=0.01; OUTER_LR=0.001; INNER_STEPS=10; TPB=4; SUP=20; QRY=20
N_OUTER=30000; VAL_EVERY=500
WORK_CKPT=f"{WORK_MODELS}/{CKPT_NAME}"; WORK_BEST=f"{WORK_MODELS}/{BEST_NAME}"
rng=np.random.RandomState(42)

model=LSTMAutoencoder(25).to(DEVICE)
opt=torch.optim.Adam(model.parameters(),lr=OUTER_LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,'min',factor=0.5,patience=5,min_lr=1e-5)
start=0; best=float('inf'); hist={'step':[],'train':[],'val':[]}

resume_path, resume_step = resolve_ckpt(CKPT_NAME)
if resume_path is not None and resume_step>0:
    ck=torch.load(resume_path, map_location=DEVICE)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    start=ck['step']; best=ck['best']; hist=ck['hist']
    print(f"RESUMED from {resume_path} @ step {start}")
else:
    print("no checkpoint found — training from scratch")

def meta_val():
    return float(np.mean([outer_step(model,opt,[c],INNER_LR,INNER_STEPS,SUP,QRY,train=False,rng=rng)
                          for c in meta_val_tasks]))

if start>=N_OUTER:
    print(f"training already complete ({start} steps). skipping to eval.")
else:
    t0=time.time()
    for step in range(start+1,N_OUTER+1):
        batch=rng.choice(meta_train_tasks,size=TPB,replace=False).tolist()
        model.train(); tl=outer_step(model,opt,batch,INNER_LR,INNER_STEPS,SUP,QRY,train=True,rng=rng)
        if step%VAL_EVERY==0:
            vl=meta_val(); sched.step(vl)
            hist['step'].append(step); hist['train'].append(tl); hist['val'].append(vl)
            print(f"step {step:6d} | train {tl:.6f} | val {vl:.6f} | {time.time()-t0:.0f}s")
            torch.save({'model':model.state_dict(),'opt':opt.state_dict(),'step':step,
                        'best':best,'hist':hist}, WORK_CKPT)     # persists via commit output
            if vl<best:
                best=vl; torch.save({'model_state_dict':model.state_dict(),'val_loss':vl,'step':step}, WORK_BEST)
    print("meta-training done. best val", best)
    json.dump(hist, open(f"{OUTPUT_PATH}/results/smap_maml_history.json","w"), indent=2)


no checkpoint found — training from scratch
step    500 | train 0.012205 | val 0.009791 | 138s
step   1000 | train 0.006350 | val 0.008176 | 274s
step   1500 | train 0.006658 | val 0.009767 | 411s
step   2000 | train 0.008503 | val 0.006686 | 547s
step   2500 | train 0.000565 | val 0.006725 | 683s
step   3000 | train 0.007650 | val 0.006826 | 819s
step   3500 | train 0.005684 | val 0.006471 | 955s
step   4000 | train 0.003993 | val 0.006232 | 1091s
step   4500 | train 0.006089 | val 0.005844 | 1227s
step   5000 | train 0.004945 | val 0.005721 | 1363s
step   5500 | train 0.003395 | val 0.006329 | 1499s
step   6000 | train 0.006152 | val 0.005624 | 1636s
step   6500 | train 0.004326 | val 0.004894 | 1772s
step   7000 | train 0.006178 | val 0.004750 | 1908s
step   7500 | train 0.006365 | val 0.004922 | 2044s
step   8000 | train 0.003891 | val 0.004982 | 2180s
step   8500 | train 0.002452 | val 0.005122 | 2317s
step   9000 | train 0.004242 | val 0.004690 | 2453s
step   9500 | train 0.00498

## 5 — Properly-trained static AE baseline (skips if already saved)

In [5]:
WORK_STATIC=f"{WORK_MODELS}/{STATIC_NAME}"
static=LSTMAutoencoder(25).to(DEVICE)
spath,_=resolve_ckpt(STATIC_NAME,'val_dummy')
if spath is not None:
    static.load_state_dict(torch.load(spath, map_location=DEVICE)['model_state_dict'])
    print("loaded existing static baseline from", spath)
else:
    pool=np.concatenate([channel_data[c]['normal_windows'] for c in meta_train_tasks],axis=0).astype(np.float32)
    p=np.random.RandomState(42).permutation(len(pool)); cut=int(0.9*len(pool))
    tr=torch.tensor(pool[p[:cut]]); va=torch.tensor(pool[p[cut:]])
    sopt=torch.optim.Adam(static.parameters(),lr=1e-3); BATCH=128; best_s=1e9; bad=0; best_state=None
    for ep in range(150):
        static.train(); pi=torch.randperm(len(tr))
        for st in range(0,len(tr),BATCH):
            b=tr[pi[st:st+BATCH]].to(DEVICE); sopt.zero_grad()
            loss=criterion(static(b),b); loss.backward(); sopt.step()
        static.eval()
        with torch.no_grad(): vl=criterion(static(va.to(DEVICE)),va.to(DEVICE)).item()
        if vl<best_s-1e-6: best_s=vl; bad=0; best_state={k:v.clone() for k,v in static.state_dict().items()}
        else:
            bad+=1
            if bad>=15: break
    static.load_state_dict(best_state)
    torch.save({'model_state_dict':static.state_dict(),'val':best_s}, WORK_STATIC)
    print("static baseline trained. best val", best_s)


static baseline trained. best val 0.005867016967386007


## 6 — Evaluation protocol (verbatim helpers), adaptation held equal

In [6]:
def adapt_model(model, support, n_steps=5, lr=0.01):
    learner=copy.deepcopy(model); learner.train(); o=torch.optim.SGD(learner.parameters(),lr=lr)
    for _ in range(n_steps):
        o.zero_grad(); l=criterion(learner(support),support); l.backward(); o.step()
    return learner
def compute_threshold(model, support, multiplier=2.0):
    model.eval()
    with torch.no_grad(): e=model.reconstruction_error(support)
    m=e.mean().item(); return m+multiplier*e.std().item() if len(e)>1 else m*1.20
def compute_metrics(scores, labels, tau):
    preds=(scores>tau).astype(int)
    if 0<preds.sum()<len(preds):
        f1=f1_score(labels,preds,zero_division=0); pr=precision_score(labels,preds,zero_division=0); rc=recall_score(labels,preds,zero_division=0)
    else: f1=pr=rc=0.0
    au=roc_auc_score(labels,scores) if len(np.unique(labels))>1 else 0.5
    return {'f1':round(float(f1),4),'precision':round(float(pr),4),'recall':round(float(rc),4),'auroc':round(float(au),4)}
def build_eval_query(channel_id, k_shot, support_seed=42, max_normal_query=100):
    r=np.random.RandomState(support_seed)
    nw=channel_data[channel_id]['normal_windows'].copy(); aw=channel_data[channel_id]['anomaly_windows'].copy()
    r.shuffle(nw); r.shuffle(aw)
    support=nw[:k_shot]; rem=nw[k_shot:k_shot+max_normal_query]
    q=np.concatenate([aw,rem],0); y=np.concatenate([np.ones(len(aw)),np.zeros(len(rem))])
    idx=r.permutation(len(q)); q,y=q[idx],y[idx]
    return (torch.tensor(support,dtype=torch.float32).to(DEVICE), torch.tensor(q,dtype=torch.float32).to(DEVICE), y)

maml_model=LSTMAutoencoder(25).to(DEVICE)
bpath,_=resolve_ckpt(BEST_NAME,'step')
maml_model.load_state_dict(torch.load(bpath, map_location=DEVICE)['model_state_dict'])
static_model=static
ADAPT_STEPS=5   # SAME test-time adaptation for MAML and Static -> isolates the meta-training effect
print("loaded MAML best from", bpath)


loaded MAML best from /kaggle/working/models/smap_maml_best.pt


## 7 — Per-channel evaluation

In [7]:
def evaluate_channel(ch, k_shot, seed):
    support,query,labels=build_eval_query(ch,k_shot,support_seed=seed); out={}
    a=adapt_model(maml_model,support,ADAPT_STEPS,0.01); a.eval()
    with torch.no_grad(): s=a.reconstruction_error(query).cpu().numpy()
    out['MAML-AE']=compute_metrics(s,labels,compute_threshold(a,support))
    a=adapt_model(static_model,support,ADAPT_STEPS,0.01); a.eval()
    with torch.no_grad(): s=a.reconstruction_error(query).cpu().numpy()
    out['Static-AE']=compute_metrics(s,labels,compute_threshold(a,support))
    a=adapt_model(MLPAutoencoder(25).to(DEVICE),support,50,0.01); a.eval()
    with torch.no_grad(): s=a.reconstruction_error(query).cpu().numpy()
    out['MLP-AE']=compute_metrics(s,labels,compute_threshold(a,support))
    sf=support.cpu().numpy().reshape(len(support),-1); qf=query.cpu().numpy().reshape(len(query),-1)
    iso=IsolationForest(n_estimators=100,contamination='auto',random_state=seed).fit(sf)
    isc=-iso.score_samples(qf); ip=(iso.predict(qf)==-1).astype(int)
    if 0<ip.sum()<len(ip):
        f1=f1_score(labels,ip,zero_division=0); pr=precision_score(labels,ip,zero_division=0); rc=recall_score(labels,ip,zero_division=0)
    else: f1=pr=rc=0.0
    au=roc_auc_score(labels,isc) if len(np.unique(labels))>1 else 0.5
    out['Isolation-Forest']={'f1':round(float(f1),4),'precision':round(float(pr),4),'recall':round(float(rc),4),'auroc':round(float(au),4)}
    return out

K_SHOTS=[1,5,10]; SEEDS=[42,123,456,789,1024]
METHODS=['MAML-AE','Static-AE','MLP-AE','Isolation-Forest']; METRICS=['f1','precision','recall','auroc']
all_results={}
for ch in EVAL_CHANNELS:
    all_results[ch]={}
    for k in K_SHOTS:
        store={m:{mt:[] for mt in METRICS} for m in METHODS}
        for seed in SEEDS:
            np.random.seed(seed); torch.manual_seed(seed)
            r=evaluate_channel(ch,k,seed)
            for m in METHODS:
                for mt in METRICS: store[m][mt].append(r[m][mt])
        all_results[ch][k]=store
    print(f"  {ch:6s} done")
json.dump(all_results, open(f"{OUTPUT_PATH}/results/smap_all_results.json","w"), indent=2)
print("evaluation complete")


  E-3    done
  D-7    done
  E-6    done
  D-6    done
  T-2    done
  A-6    done
  D-3    done
evaluation complete


## 8 — Macro results: MAML vs properly-trained Static

In [8]:
print(f"{'shot':>4} | " + " | ".join(f"{m:>18s}" for m in METHODS))
for k in K_SHOTS:
    c=[]
    for m in METHODS:
        f1=np.mean([np.mean(all_results[ch][k][m]['f1'])    for ch in EVAL_CHANNELS])
        au=np.mean([np.mean(all_results[ch][k][m]['auroc']) for ch in EVAL_CHANNELS])
        c.append(f"F1 {f1:.3f}/AU {au:.3f}")
    print(f"{k:>4} | " + " | ".join(f"{x:>18s}" for x in c))
print("\nMAML-AE minus Static-AE (macro):")
for k in K_SHOTS:
    for mt in ['f1','auroc']:
        a=np.mean([np.mean(all_results[ch][k]['MAML-AE'][mt])  for ch in EVAL_CHANNELS])
        b=np.mean([np.mean(all_results[ch][k]['Static-AE'][mt]) for ch in EVAL_CHANNELS])
        print(f"  {k:2d}-shot {mt:5s}: {a-b:+.4f}")


shot |            MAML-AE |          Static-AE |             MLP-AE |   Isolation-Forest
   1 |  F1 0.316/AU 0.487 |  F1 0.279/AU 0.450 |  F1 0.391/AU 0.586 |  F1 0.000/AU 0.500
   5 |  F1 0.152/AU 0.475 |  F1 0.134/AU 0.453 |  F1 0.172/AU 0.586 |  F1 0.161/AU 0.590
  10 |  F1 0.107/AU 0.480 |  F1 0.102/AU 0.457 |  F1 0.145/AU 0.589 |  F1 0.265/AU 0.607

MAML-AE minus Static-AE (macro):
   1-shot f1   : +0.0370
   1-shot auroc: +0.0369
   5-shot f1   : +0.0187
   5-shot auroc: +0.0217
  10-shot f1   : +0.0049
  10-shot auroc: +0.0227
